# 04 — Demand Forecasting

**AI Smart Inventory Management & Demand Forecasting System**  
**Day 9 — Phase 5: Model Building & Forecasting**

---

## Objectives

1. Load and validate processed sales + products data
2. Engineer predictive features (lags, rolling averages, calendar)
3. Temporal train / validation split
4. Train Baseline model (30-day Simple Moving Average)
5. Train Champion model (Random Forest Regressor)
6. Evaluate with MAE and RMSE — select champion
7. Generate 30-day demand forecasts for every SKU
8. Compute safety stock and recommended reorder points
9. Persist forecasts to SQLite and CSV
10. Summary and conclusions

---

## Section 1 — Data Loading & Validation

In [ ]:
import os
import sys
import uuid
import sqlite3
import warnings
from datetime import datetime, timedelta, timezone

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)

# Paths
NOTEBOOK_DIR = os.path.dirname(os.path.abspath('__file__'))
PROJECT_ROOT = os.path.abspath(os.path.join(NOTEBOOK_DIR, '..', '..'))
DATA_DIR = os.path.join(PROJECT_ROOT, 'data', 'processed')
DB_PATH = os.path.join(PROJECT_ROOT, 'data', 'inventory.db')

print(f'Project Root: {PROJECT_ROOT}')
print(f'Data Dir:     {DATA_DIR}')

In [ ]:
# Load datasets
sales = pd.read_csv(os.path.join(DATA_DIR, 'sales_processed.csv'), parse_dates=['sale_date'])
products = pd.read_csv(os.path.join(DATA_DIR, 'products_processed.csv'))

print(f'Sales:    {sales.shape[0]:,} rows x {sales.shape[1]} cols')
print(f'Products: {products.shape[0]:,} rows x {products.shape[1]} cols')
print()
print('--- Sales Dataset ---')
print(f'Date range: {sales["sale_date"].min().date()} to {sales["sale_date"].max().date()}')
print(f'Unique products: {sales["product_id"].nunique()}')
print(f'Missing values: {sales.isnull().sum().sum()}')
print()
sales.head()

---

## Section 2 — Feature Engineering

In [ ]:
# Configuration
LAG_FEATURES = [1, 7, 14, 30]
ROLLING_WINDOWS = [7, 14, 30]
VALIDATION_DAYS = 60
FORECAST_HORIZON = 30
SMA_WINDOW = 30
Z_SCORE = 1.645  # 95% service level

FEATURE_COLS = (
    [f'lag_{d}' for d in LAG_FEATURES]
    + [f'rolling_mean_{w}' for w in ROLLING_WINDOWS]
    + [f'rolling_std_{w}' for w in ROLLING_WINDOWS]
    + ['day_of_week', 'month', 'quarter', 'is_weekend']
)
print(f'Feature columns ({len(FEATURE_COLS)}):')
for f in FEATURE_COLS:
    print(f'  - {f}')

In [ ]:
# Build daily demand time series per product
daily = (
    sales
    .groupby(['product_id', 'sale_date'])
    .agg(daily_qty=('quantity', 'sum'))
    .reset_index()
)

# Fill calendar gaps with 0-demand days
date_min = daily['sale_date'].min()
date_max = daily['sale_date'].max()
all_dates = pd.date_range(date_min, date_max, freq='D')
product_ids = daily['product_id'].unique()

idx = pd.MultiIndex.from_product([product_ids, all_dates],
                                  names=['product_id', 'sale_date'])
full = pd.DataFrame(index=idx).reset_index()
full = full.merge(daily, on=['product_id', 'sale_date'], how='left')
full['daily_qty'] = full['daily_qty'].fillna(0).astype(float)

n_products = full['product_id'].nunique()
n_days = (date_max - date_min).days + 1
print(f'{n_products} products x {n_days} days = {full.shape[0]:,} rows')

In [ ]:
# Engineer lag, rolling, and calendar features
full = full.sort_values(['product_id', 'sale_date']).copy()

for lag in LAG_FEATURES:
    full[f'lag_{lag}'] = full.groupby('product_id')['daily_qty'].shift(lag)

for window in ROLLING_WINDOWS:
    full[f'rolling_mean_{window}'] = full.groupby('product_id')['daily_qty'].transform(
        lambda s: s.shift(1).rolling(window, min_periods=1).mean()
    )
    full[f'rolling_std_{window}'] = full.groupby('product_id')['daily_qty'].transform(
        lambda s: s.shift(1).rolling(window, min_periods=1).std()
    ).fillna(0)

full['day_of_week'] = full['sale_date'].dt.dayofweek
full['month'] = full['sale_date'].dt.month
full['quarter'] = full['sale_date'].dt.quarter
full['is_weekend'] = (full['day_of_week'] >= 5).astype(int)

print('Feature engineering complete.')
full[FEATURE_COLS + ['daily_qty']].describe().round(3)

---

## Section 3 — Temporal Train / Validation Split

In [ ]:
cutoff = full['sale_date'].max() - pd.Timedelta(days=VALIDATION_DAYS)
ready = full.dropna(subset=[c for c in FEATURE_COLS if c in full.columns])

train = ready[ready['sale_date'] <= cutoff].copy()
val = ready[ready['sale_date'] > cutoff].copy()

print(f'Temporal cutoff: {cutoff.date()}')
print(f'Train set: {train.shape[0]:,} rows  ({train["product_id"].nunique()} products)')
print(f'Val set:   {val.shape[0]:,} rows  ({val["product_id"].nunique()} products)')
print(f'Train date range: {train["sale_date"].min().date()} to {train["sale_date"].max().date()}')
print(f'Val date range:   {val["sale_date"].min().date()} to {val["sale_date"].max().date()}')

---

## Section 4 — Baseline Model (Simple Moving Average)

In [ ]:
# Baseline: 30-day SMA per product from training data
latest_train = train.sort_values('sale_date').groupby('product_id').tail(SMA_WINDOW)
sma = latest_train.groupby('product_id')['daily_qty'].mean().rename('sma_pred')

sma_preds = val[['product_id', 'sale_date', 'daily_qty']].copy()
sma_preds = sma_preds.merge(sma, on='product_id', how='left')
sma_preds['sma_pred'] = sma_preds['sma_pred'].fillna(0)

sma_mae = mean_absolute_error(sma_preds['daily_qty'], sma_preds['sma_pred'])
sma_rmse = np.sqrt(mean_squared_error(sma_preds['daily_qty'], sma_preds['sma_pred']))

print(f'Baseline (SMA-{SMA_WINDOW}) Results:')
print(f'  MAE  = {sma_mae:.4f}')
print(f'  RMSE = {sma_rmse:.4f}')

---

## Section 5 — Random Forest Model

In [ ]:
# Train Random Forest Regressor
X_train = train[FEATURE_COLS].values
y_train = train['daily_qty'].values
X_val = val[FEATURE_COLS].values
y_val = val['daily_qty'].values

rf = RandomForestRegressor(
    n_estimators=200,
    max_depth=12,
    random_state=42,
    n_jobs=-1,
)
rf.fit(X_train, y_train)
rf_preds = np.clip(rf.predict(X_val), 0, None)

rf_mae = mean_absolute_error(y_val, rf_preds)
rf_rmse = np.sqrt(mean_squared_error(y_val, rf_preds))

print(f'Random Forest Results:')
print(f'  MAE  = {rf_mae:.4f}')
print(f'  RMSE = {rf_rmse:.4f}')

---

## Section 6 — Model Comparison

In [ ]:
comparison = pd.DataFrame([
    {'Model': f'Baseline (SMA-{SMA_WINDOW})', 'MAE': round(sma_mae, 4), 'RMSE': round(sma_rmse, 4)},
    {'Model': 'Random Forest Regressor', 'MAE': round(rf_mae, 4), 'RMSE': round(rf_rmse, 4)},
])
comparison['Champion'] = comparison['RMSE'] == comparison['RMSE'].min()

print('=== Model Comparison ===')
print(comparison.to_string(index=False))
print()

champion_name = 'Random Forest' if rf_rmse <= sma_rmse else 'SMA'
print(f'** Champion Model: {champion_name} **')

# Feature Importances
if champion_name == 'Random Forest':
    importances = pd.Series(rf.feature_importances_, index=FEATURE_COLS).sort_values(ascending=False)
    print('\nTop 5 Feature Importances:')
    for feat, imp in importances.head(5).items():
        print(f'  {feat:25s}  {imp:.4f}')

In [ ]:
# Visualize feature importances
fig, ax = plt.subplots(figsize=(10, 6))
importances_sorted = importances.sort_values(ascending=True)
colors = plt.cm.viridis(np.linspace(0.3, 0.9, len(importances_sorted)))
ax.barh(importances_sorted.index, importances_sorted.values, color=colors)
ax.set_xlabel('Feature Importance')
ax.set_title('Random Forest Feature Importances')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, 'analytics', 'visuals', 'forecasts', 'F03_feature_importances.png'),
            dpi=150, bbox_inches='tight')
plt.show()
print('Saved: F03_feature_importances.png')

---

## Section 7 — 30-Day Future Forecasts

In [ ]:
# Recursive multi-step forecasting
last_date = full['sale_date'].max()
forecast_rows = []

for pid in full['product_id'].unique():
    prod_data = full[full['product_id'] == pid].sort_values('sale_date')
    history = prod_data['daily_qty'].values.tolist()
    recent = history[-max(LAG_FEATURES + ROLLING_WINDOWS):]

    for step in range(1, FORECAST_HORIZON + 1):
        fdate = last_date + pd.Timedelta(days=step)
        features = {}
        for lag in LAG_FEATURES:
            features[f'lag_{lag}'] = recent[-lag] if len(recent) >= lag else 0
        for w in ROLLING_WINDOWS:
            window_vals = recent[-w:] if len(recent) >= w else recent
            features[f'rolling_mean_{w}'] = np.mean(window_vals) if window_vals else 0
            features[f'rolling_std_{w}'] = np.std(window_vals) if len(window_vals) > 1 else 0
        features['day_of_week'] = fdate.dayofweek
        features['month'] = fdate.month
        features['quarter'] = fdate.quarter
        features['is_weekend'] = int(fdate.dayofweek >= 5)

        X_row = np.array([[features[c] for c in FEATURE_COLS]])
        pred = max(0, rf.predict(X_row)[0])
        recent.append(pred)

        forecast_rows.append({
            'product_id': pid,
            'forecast_date': fdate.strftime('%Y-%m-%d'),
            'predicted_daily_qty': round(pred, 4),
        })

forecasts_df = pd.DataFrame(forecast_rows)
print(f'Generated {len(forecasts_df):,} daily forecast rows')
print(f'Date range: {forecasts_df["forecast_date"].min()} to {forecasts_df["forecast_date"].max()}')

# Aggregate to 30-day totals per SKU
agg = (
    forecasts_df
    .groupby('product_id')
    .agg(
        predicted_qty=('predicted_daily_qty', 'sum'),
        avg_daily_forecast=('predicted_daily_qty', 'mean'),
        std_daily_forecast=('predicted_daily_qty', 'std'),
    )
    .reset_index()
)
agg['predicted_qty'] = agg['predicted_qty'].round(0).astype(int)
agg['avg_daily_forecast'] = agg['avg_daily_forecast'].round(4)
agg['std_daily_forecast'] = agg['std_daily_forecast'].fillna(0).round(4)

# Merge product info
prod_cols = ['product_id', 'sku', 'name', 'category', 'lead_time_days',
             'current_stock_quantity', 'reorder_point']
available = [c for c in prod_cols if c in products.columns]
agg = agg.merge(products[available], on='product_id', how='left')

print(f'\nTotal predicted demand (all SKUs): {agg["predicted_qty"].sum():,} units')
agg.head(10)

---

## Section 8 — Safety Stock & Reorder Points

In [ ]:
# Safety Stock = Z * sigma_daily * sqrt(lead_time)
# Reorder Point = avg_daily_demand * lead_time + safety_stock

lead = agg['lead_time_days'].fillna(7).astype(float)
sigma = agg['std_daily_forecast'].fillna(0).astype(float)
avg_demand = agg['avg_daily_forecast'].fillna(0).astype(float)

agg['safety_stock'] = (Z_SCORE * sigma * np.sqrt(lead)).round(0).astype(int)
agg['rec_reorder_point'] = (avg_demand * lead + agg['safety_stock']).round(0).astype(int)
agg['rec_reorder_qty'] = (avg_demand * lead * 2 + agg['safety_stock']).round(0).astype(int)

print('Safety Stock & Reorder Point Summary:')
print(f'  Average safety stock:     {agg["safety_stock"].mean():.1f} units')
print(f'  Average reorder point:    {agg["rec_reorder_point"].mean():.1f} units')
print(f'  Average reorder quantity: {agg["rec_reorder_qty"].mean():.1f} units')
print()

# Display top 10 by predicted demand
top10 = agg.nlargest(10, 'predicted_qty')[['sku', 'name', 'category', 'predicted_qty',
                                             'safety_stock', 'rec_reorder_point', 'rec_reorder_qty',
                                             'current_stock_quantity']]
print('Top 10 SKUs by Predicted 30-Day Demand:')
top10

In [ ]:
# Visualization: Top 15 SKUs by predicted demand
fig, ax = plt.subplots(figsize=(12, 7))
top15 = agg.nlargest(15, 'predicted_qty').sort_values('predicted_qty', ascending=True)
colors = plt.cm.plasma(np.linspace(0.2, 0.9, len(top15)))
bars = ax.barh(top15['sku'], top15['predicted_qty'], color=colors, edgecolor='white', linewidth=0.5)
ax.set_xlabel('Predicted 30-Day Demand (units)')
ax.set_title('Top 15 SKUs by Forecasted 30-Day Demand')
ax.grid(axis='x', alpha=0.3)
for bar, val in zip(bars, top15['predicted_qty']):
    ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
            f'{val:,}', va='center', fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, 'analytics', 'visuals', 'forecasts', 'F04_top15_demand_forecast.png'),
            dpi=150, bbox_inches='tight')
plt.show()
print('Saved: F04_top15_demand_forecast.png')

In [ ]:
# Visualization: Safety Stock vs Current Stock
fig, ax = plt.subplots(figsize=(12, 7))
plot_data = agg.sort_values('safety_stock', ascending=False).head(20)
x = np.arange(len(plot_data))
width = 0.35
ax.bar(x - width/2, plot_data['current_stock_quantity'], width, label='Current Stock', color='#4CAF50', alpha=0.8)
ax.bar(x + width/2, plot_data['rec_reorder_point'], width, label='Reorder Point', color='#FF5722', alpha=0.8)
ax.set_xlabel('SKU')
ax.set_ylabel('Quantity (units)')
ax.set_title('Current Stock vs Recommended Reorder Point (Top 20 by Safety Stock)')
ax.set_xticks(x)
ax.set_xticklabels(plot_data['sku'], rotation=45, ha='right', fontsize=8)
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, 'analytics', 'visuals', 'forecasts', 'F05_stock_vs_reorder.png'),
            dpi=150, bbox_inches='tight')
plt.show()
print('Saved: F05_stock_vs_reorder.png')

---

## Section 9 — Database Persistence

In [ ]:
# NOTE: The train_forecast.py script has already persisted data.
# This cell demonstrates how to verify the database contents.

if os.path.exists(DB_PATH):
    conn = sqlite3.connect(DB_PATH)
    db_forecasts = pd.read_sql_query('SELECT * FROM forecasts', conn)
    conn.close()
    
    print(f'Forecasts in database: {len(db_forecasts)} rows')
    print(f'Columns: {list(db_forecasts.columns)}')
    print()
    db_forecasts.head()
else:
    print(f'Database not found at {DB_PATH}')

In [ ]:
# Verify CSV export
csv_path = os.path.join(DATA_DIR, 'forecasts_processed.csv')
if os.path.exists(csv_path):
    csv_forecasts = pd.read_csv(csv_path)
    print(f'Forecasts CSV: {len(csv_forecasts)} rows x {csv_forecasts.shape[1]} cols')
    print(f'Columns: {list(csv_forecasts.columns)}')
    print()
    csv_forecasts.head()
else:
    print(f'CSV not found at {csv_path}')

---

## Section 10 — Summary & Conclusions

In [ ]:
print('=' * 70)
print('  DEMAND FORECASTING SUMMARY')
print('=' * 70)
print()
print(f'  Products forecasted     : {n_products}')
print(f'  Training set size       : {train.shape[0]:,} daily observations')
print(f'  Validation set size     : {val.shape[0]:,} daily observations')
print(f'  Feature count           : {len(FEATURE_COLS)}')
print(f'  Forecast horizon        : {FORECAST_HORIZON} days')
print()
print('  MODEL EVALUATION')
print(f'  -----------------------------------------------')
print(f'  Baseline (SMA-{SMA_WINDOW})   MAE={sma_mae:.4f}  RMSE={sma_rmse:.4f}')
print(f'  Random Forest         MAE={rf_mae:.4f}  RMSE={rf_rmse:.4f}')
print(f'  -----------------------------------------------')
print(f'  Champion: {champion_name}')
print()
print(f'  FORECAST RESULTS')
print(f'  Total 30-day demand     : {agg["predicted_qty"].sum():,} units')
print(f'  Avg safety stock        : {agg["safety_stock"].mean():.1f} units')
print(f'  Avg reorder point       : {agg["rec_reorder_point"].mean():.1f} units')
print(f'  DB records              : {len(db_forecasts) if os.path.exists(DB_PATH) else "N/A"}')
print(f'  CSV records             : {len(csv_forecasts) if os.path.exists(csv_path) else "N/A"}')
print()
print('  KEY INSIGHTS')
print('  1. Random Forest narrowly beats SMA baseline on RMSE')
print('  2. day_of_week and rolling_std_30 are the strongest predictors')
print('  3. Demand is sparse (many zero-demand days) which limits model gains')
print('  4. Safety stock calculations at 95% service level are conservative')
print('  5. Reorder points incorporate both demand uncertainty and lead time')
print()
print('=' * 70)
print('  Phase 5 Complete -- Model Building & Forecasting')
print('=' * 70)